<img src="../../shared/alchemi-banner-left.png" alt="NVIDIA ALCHEMI: AI for Chemistry and Materials Science" style="display:block;box-sizing:border-box;width:100%;max-width:100%;height:auto;">

# 04 · Hooks

**Goal:** Attach observation, safety, logging, and saved-state behavior to a running workflow through public hook APIs.

**Core concepts:** `Hook` is a structural protocol. `DynamicsStage` names callback boundaries. `DynamicsContext` carries the state a host exposes at one callback.

The [Core playbook](../00-core-playbook/alchemi-core-playbook.ipynb) used model-provided neighbors, `NaNDetectorHook`, snapshots, and a one-step observer. Here you will inspect their registration and run lifecycle.

You will reuse [`AtomicData` and `Batch` from Part 01](../01-atomicdata-batch/atomicdata-and-batch.ipynb) and the [configured model boundary from Part 03](../03-model-interfaces-composition/model-interfaces-composition.ipynb). We start with a four-step argon host, then apply one small observer and a safety guard to the checked molecular batch.

<details>
<summary>Where NVIDIA ALCHEMI fits (recap)</summary>

[ALCHEMI](https://developer.nvidia.com/cuda/cuda-x-libraries/alchemi) brings together Python building blocks, accelerated kernels, and deployable services for atomistic workflows.

- **ALCHEMI Toolkit** is a GPU-first Python framework with a unified, composable API for MLIPs and custom models. It provides GPU-native atomic data and batching, model adapters, MD classes, hooks, model training,  and single- to multi-GPU pipelines.   
[GitHub repo](https://github.com/NVIDIA/nvalchemi-toolkit) · [Docs](https://nvidia.github.io/nvalchemi-toolkit/) · Apache 2.0 license

- **Toolkit-Ops** supplies GPU-optimized, batched operations for neighbor lists, dynamics, dispersion, and electrostatics, with PyTorch and JAX bindings.    
[GitHub repo](https://github.com/NVIDIA/nvalchemi-toolkit-ops) · [Docs](https://nvidia.github.io/nvalchemi-toolkit-ops/) · Apache 2.0 license

- **ALCHEMI NIM microservices** package supported atomistic workflows as cloud-ready services. The current catalog includes Batched Geometry Relaxation for structural optimization and Batched Molecular Dynamics.  Self-hosting uses an NVIDIA AI Enterprise license.   
 [Transparency card](https://docs.nvidia.com/nim/alchemi/alchemi-bgr/1.0.0/ai-transparency-card/overview.html) · [Docs](https://docs.nvidia.com/nim/alchemi/alchemi-bgr/latest/index.html)
</details>

## Where hooks fit

<object data="../../shared/curriculum-map-04.svg" type="image/svg+xml" style="display:block;box-sizing:border-box;width:100%;max-width:100%;height:auto;" aria-label="ALCHEMI Toolkit curriculum. Select an available lesson to open its notebook.">
  <img src="../../shared/curriculum-map-04.svg" alt="ALCHEMI Toolkit curriculum. Part 04 teaches hooks for reusable behavior at named workflow stages. Earlier lessons cover atomic data, batching, data loading, and model composition. Later lessons cover base dynamics, GPU pipelines and profiling, training, and domain decomposition for multi-GPU execution of very large systems." style="display:block;box-sizing:border-box;width:100%;max-width:100%;height:auto;">
</object>

Green marks Part 04. Hooks add behavior at named workflow stages. Part 05 owns relaxation strategy and `BaseDynamics`.

In [ ]:
import inspect

import helpers
import nvalchemi.dynamics.hooks as dynamics_hooks
import pandas as pd
import torch
from ase import Atoms
from nvalchemi.data import AtomicData, Batch
from nvalchemi.dynamics import FIRE2, ConvergenceHook, DynamicsStage, HostMemory
from nvalchemi.hooks import DynamicsContext, Hook
from nvalchemi.models import AIMNet2Wrapper
from nvalchemi.models.lj import LennardJonesModelWrapper

helpers.configure_presentation()

In [ ]:
LJ_EPSILON = 0.0104  # eV
LJ_SIGMA = 3.40  # Å
LJ_CUTOFF = 8.5  # Å
LJ_R_MIN = 2 ** (1 / 6) * LJ_SIGMA

## Start with a four-step argon host

Four argon atoms near the Lennard-Jones pair minimum make a fast, deterministic host for inspecting built-ins. The open cluster is an API probe, not bulk argon or an equilibrated structure. Four `FIRE2` steps do not support a relaxation or performance claim.

In [ ]:
argon_atoms = Atoms(
    "Ar4",
    positions=[[0, 0, 0], [LJ_R_MIN, 0, 0], [0, LJ_R_MIN, 0], [0, 0, LJ_R_MIN]],
)
argon_graph = AtomicData.from_atoms(argon_atoms, device="cpu")

In [ ]:
argon_graph.add_system_property("energy", torch.zeros(1, 1))
argon_graph.add_node_property("forces", torch.zeros(4, 3))
argon_graph.add_system_property("status", torch.zeros(1, 1, dtype=torch.long))
argon_batch = Batch.from_data_list([argon_graph], device="cpu")
argon_initial = argon_batch.clone()

## Ask the model to maintain neighbors

An iterative host changes positions, so its neighbor representation can become stale. `make_neighbor_hooks()` is the model-owned public path for keeping that derived input current. The returned hook runs at `BEFORE_COMPUTE`, before each model call.

In [ ]:
lj_model = LennardJonesModelWrapper(
    epsilon=LJ_EPSILON, sigma=LJ_SIGMA, cutoff=LJ_CUTOFF
).eval()
model = lj_model

In [ ]:
{
    "active outputs": sorted(model.model_config.active_outputs),
    "neighbor cutoff (Å)": model.model_config.neighbor_config.cutoff,
    "device": str(argon_batch.device),
}

## Register safety, logging, and snapshots

Each built-in owns one job:

- `MaxForceClampHook` bounds force-vector magnitudes after compute;
- `NaNDetectorHook` stops on non-finite energy or forces;
- `LoggingHook` copies scalar rows for later inspection; and
- `SnapshotHook` writes complete batch states to a public data sink.

The force limit below is deliberately small so its effect appears in four steps. It is not a recommended physical setting. Frequent clamping calls for a better structure, model, or timestep.

In [ ]:
lj_log_rows: list[dict[str, float]] = []

def collect_lj_rows(_step: int, rows: list[dict[str, float]]) -> None:
    lj_log_rows.extend(rows)

In [ ]:
force_clamp = dynamics_hooks.MaxForceClampHook(max_force=0.003, frequency=1)
nan_guard = dynamics_hooks.NaNDetectorHook(frequency=1)

`force_clamp` and `nan_guard` both run at `AFTER_COMPUTE`. Hooks at the same stage run in registration order, so the detector reads the bounded forces. Clamping cannot repair NaN or infinity; the detector still stops those failures.

In [ ]:
lj_logger = dynamics_hooks.LoggingHook(
    backend="custom", writer_fn=collect_lj_rows, frequency=1
)

In [ ]:
snapshot_sink = HostMemory(capacity=4)
snapshot_hook = dynamics_hooks.SnapshotHook(sink=snapshot_sink, frequency=2)

In [ ]:
neighbor_hooks = model.make_neighbor_hooks()

In [ ]:
lj_hooks = [*neighbor_hooks, force_clamp, nan_guard, lj_logger, snapshot_hook]
lj_host = FIRE2(
    model=lj_model,
    dt=0.01,
    n_steps=4,
    convergence_hook=None,
    hooks=lj_hooks,
)

<div style="display:block;box-sizing:border-box;width:100%;max-width:100%;min-width:0;background:#111619;color:#F3F4F6;border:1px solid #2E353B;border-radius:10px;padding:0.9rem 1rem;line-height:1.45;overflow:hidden;overflow-wrap:anywhere;">
  <div style="color:#76B900;font-size:0.72rem;font-weight:700;letter-spacing:0.055em;margin-bottom:0.45rem;">ALCHEMI TOOLKIT API</div>
  <code style="display:block;color:#FFFFFF;font-size:1.02rem;font-weight:650;white-space:normal;overflow-wrap:anywhere;">FIRE2(..., hooks=lj_hooks) -&gt; registered host</code>
  <div style="border-top:1px solid #2E353B;margin-top:0.7rem;padding-top:0.65rem;color:#CDD2D8;font-size:0.92rem;">
    <div><strong style="color:#F3F4F6;">Input</strong> · hook objects in user-owned registration order.</div>
    <div style="margin-top:0.25rem;"><strong style="color:#F3F4F6;">Result</strong> · a host that creates callback contexts and enters and exits hook resources once per run.</div>
  </div>
</div>

Construction validates each hook's stage and positive integer frequency. `run()` owns optional hook resource setup and cleanup. Cleanup also runs after an exception and does not mean the workflow succeeded.

In [ ]:
pd.DataFrame(
    {
        "hook": [type(hook).__name__ for hook in lj_host.hooks],
        "stage": [hook.stage.name for hook in lj_host.hooks],
        "frequency": [hook.frequency for hook in lj_host.hooks],
    }
)

### Run the ordinary host path

The host starts with `step_count=0`. Registry dispatch uses `step_count % frequency == 0`, so logging should record steps 0 through 3 and snapshots should capture steps 0 and 2. `LoggingHook` closes its background writer before `run()` returns, which makes the rows safe to inspect immediately.

In [ ]:
argon_result = lj_host.run(argon_batch)

In [ ]:
pd.DataFrame(lj_log_rows)

In [ ]:
snapshot_batch = snapshot_sink.read()

In [ ]:
assert len(lj_log_rows) == 4
assert snapshot_batch.num_graphs == 2
{
    "completed steps": lj_host.step_count,
    "logged rows": len(lj_log_rows),
    "saved complete states": snapshot_batch.num_graphs,
}

In [ ]:
{
    "finite energy": bool(torch.isfinite(argon_result.energy).all()),
    "finite forces": bool(torch.isfinite(argon_result.forces).all()),
    "max force (eV/Å)": float(argon_result.forces.norm(dim=-1).max()),
    "status": argon_result.status.cpu().reshape(-1).tolist(),
}

## Watch a real safety failure

An Ar pair separated by 0.001 Å drives the Lennard-Jones repulsion beyond finite `float32` values. A separate one-step `FIRE2` host registers `NaNDetectorHook` normally. The host creates the callback context and closes hook resources when the detector raises.

In [ ]:
failure_graph = AtomicData(
    positions=torch.tensor([[0.0, 0.0, 0.0], [0.0, 0.0, 0.001]]),
    atomic_numbers=torch.tensor([18, 18]),
    forces=torch.zeros(2, 3),
    energy=torch.zeros(1, 1),
)
failure_graph.add_system_property("status", torch.zeros(1, 1, dtype=torch.long))
failure_batch = Batch.from_data_list([failure_graph])
failure_host = FIRE2(
    model=lj_model, dt=0.01, n_steps=1, convergence_hook=None,
    hooks=[*lj_model.make_neighbor_hooks(), dynamics_hooks.NaNDetectorHook(frequency=1)],
)

In [ ]:
try:
    failure_host.run(failure_batch)
except RuntimeError as error:
    print(f"{type(error).__name__}: {error}")

## Follow one dynamics step

The host owns this sequence. Solid arrows show normal control flow; the dotted edge runs only when the convergence evaluator returns graph indices.

```mermaid
flowchart TB
    enter["run(): enter hook resources once"] --> before["BEFORE_STEP"]
    before --> pre["BEFORE_PRE_UPDATE → pre_update() → AFTER_PRE_UPDATE"]
    pre --> compute["BEFORE_COMPUTE → compute() → AFTER_COMPUTE"]
    compute --> post["BEFORE_POST_UPDATE → post_update() → AFTER_POST_UPDATE"]
    post --> after["AFTER_STEP"]
    after --> evaluate["convergence_hook.evaluate(batch)"]
    evaluate -. "indices returned" .-> converge["ON_CONVERGE"]
    evaluate --> increment["increment step_count"]
    converge --> increment
    increment --> more{"more steps?"}
    more -- yes --> before
    more -- no --> exit["exit hook resources once"]
```

`DynamicsStage` names the nine callback boundaries in the boxes. `pre_update()`, `compute()`, `post_update()`, convergence evaluation, and the loop belong to the host.

In [ ]:
pd.DataFrame(
    [(stage.value, stage.name) for stage in DynamicsStage],
    columns=["value", "stage"],
).set_index("value")

In [ ]:
inspect.signature(DynamicsContext)

In [ ]:
{
    "NaN guard satisfies Hook": isinstance(nan_guard, Hook),
    "registered host": type(lj_host).__name__,
    "registered hooks": len(lj_host.hooks),
}

## Separate convergence from status migration

An object passed through `convergence_hook=` supplies `evaluate(batch)` directly to the host after registered `AFTER_STEP` hooks. The host checks it every step, uses returned graph indices for `ON_CONVERGE`, and can stop when every graph converges. Its `stage` and `frequency` do not gate this direct call in Toolkit 0.2.

A `ConvergenceHook` placed in `hooks=[...]` follows ordinary registry dispatch. With `source_status` and `target_status`, it can change matching values in `batch.status`. That callback result does not feed the host's convergence decision.

In [ ]:
evaluator = ConvergenceHook.from_fmax(threshold=1.0, frequency=99)
evaluator_host = FIRE2(
    model=lj_model, dt=0.01, n_steps=3, convergence_hook=evaluator,
    hooks=lj_model.make_neighbor_hooks(),
)
evaluator_batch = evaluator_host.run(argon_initial.clone())

The evaluator stopped the host after step 0 even though its `frequency` was 99. Its direct `evaluate(batch)` role left `batch.status` at zero. Now register a second `ConvergenceHook` only through `hooks=[...]`.

In [ ]:
status_hook = ConvergenceHook.from_fmax(
    threshold=1.0, source_status=0, target_status=1, frequency=1
)
status_host = FIRE2(
    model=lj_model, dt=0.01, n_steps=3, convergence_hook=None,
    hooks=[*lj_model.make_neighbor_hooks(), status_hook],
)
status_batch = status_host.run(argon_initial.clone())
pd.DataFrame(
    {
        "role": ["convergence evaluator", "registered status hook"],
        "completed steps": [evaluator_host.step_count, status_host.step_count],
        "final status": [int(evaluator_batch.status[0, 0]), int(status_batch.status[0, 0])],
    }
)

`Batch` owns the system-level `status` tensor. The direct evaluator returns indices to the host and leaves that tensor alone. The registered status hook writes `1`, then the host completes all three requested loop iterations because no direct evaluator is present.

The molecular observer below follows a third ownership pattern: it reads the active batch from `DynamicsContext` and copies selected scalar values into its own Python list. It does not change positions, forces, energy, or status.

## Observe real molecules

Ethyne, phenol, and 2,3-dimethylbutane come from the curated [NCI Atlas](https://github.com/Honza-R/NCIAtlas) subset ([CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)). The supplied `aimnet2-wb97m-d3_0` checkpoint is distributed by AIMNet under the MIT license. We request molecular energy in eV and atom forces in eV/Å.

A four-step `FIRE2` host demonstrates observation and non-finite safety on three different graph sizes. It does not establish geometry convergence, relaxation quality, model accuracy, or performance.

In [ ]:
molecular_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
molecule_labels = ("Ethyne", "Phenol", "2,3-dimethylbutane")
molecules, molecule_table = helpers.load_molecule_selection(molecule_labels)
molecule_table.set_index("system_id")

### Rebuild the model input

Each molecule becomes one `AtomicData` graph. `Batch` owns the packed positions, model outputs, stable `system_id`, and workflow `status`. The initial status is zero for every graph; this host has no status-migration hook.

In [ ]:
molecular_graphs = [
    AtomicData.from_atoms(atoms, device=molecular_device) for atoms in molecules
]
for graph in molecular_graphs:
    graph.add_system_property("energy", torch.zeros((1, 1), device=molecular_device))
    graph.add_node_property("forces", torch.zeros((graph.num_nodes, 3), device=molecular_device))

In [ ]:
for graph, charge in zip(molecular_graphs, molecule_table["charge"], strict=True):
    graph.add_system_property("charge", torch.tensor([[charge]], device=molecular_device, dtype=torch.float32))
    graph.add_system_property("status", torch.zeros((1, 1), device=molecular_device, dtype=torch.long))
molecular_batch = Batch.from_data_list(molecular_graphs, device=molecular_device)
molecular_batch.add_key("system_id", [torch.tensor(i, device=molecular_device) for i in range(3)], level="system")

### Use the supplied model boundary

The helper verifies the `aimnet2-wb97m-d3_0` checkpoint checksum before the public wrapper loads it. AIMNet2 supports the neutral organic elements in these three examples. We freeze model parameters while preserving position gradients needed for forces.

The wrapper returns graph energy with shape `[B, 1]` in eV and atom forces with shape `[V, 3]` in eV/Å. This interface check does not establish chemical accuracy outside the checkpoint's documented scope.

In [ ]:
model = AIMNet2Wrapper.from_checkpoint(
    helpers.model_checkpoint(), device=molecular_device, compile_model=False
).eval()
model = helpers.freeze_model(model)

In [ ]:
model.set_config("active_outputs", {"energy", "forces"})

In [ ]:
molecular_neighbor_hooks = model.make_neighbor_hooks()

In [ ]:
pd.DataFrame(
    {
        "hook": [type(hook).__name__ for hook in molecular_neighbor_hooks],
        "stage": [hook.stage.name for hook in molecular_neighbor_hooks],
        "frequency": [hook.frequency for hook in molecular_neighbor_hooks],
    }
)

## Write one small observer

`EnergyHistoryHook` satisfies `Hook` structurally. It declares `AFTER_COMPUTE`, stores a positive frequency, and implements `__call__(ctx, stage)`. Optional registration and context-manager methods make the host-owned lifecycle visible.

<div style="display:block;box-sizing:border-box;width:100%;max-width:100%;min-width:0;background:#F2F3F1;color:#1B1E20;border:1px solid #D6D9D4;border-radius:8px;padding:0.78rem 0.95rem;line-height:1.45;overflow:hidden;overflow-wrap:anywhere;">
  <div style="color:#725B22;font-size:0.78rem;font-weight:700;letter-spacing:0.03em;margin-bottom:0.25rem;">💡 Highlight</div>
  <div style="min-width:0;font-size:0.98rem;overflow-wrap:anywhere;">The host builds each <code style="background:#E0E3DE;color:#111315;border:1px solid #CDD1CB;border-radius:4px;padding:0.05rem 0.28rem;font-weight:650;white-space:normal;overflow-wrap:anywhere;">DynamicsContext</code>. Read the active state during the callback, then keep only the detached values your observer owns.</div>
</div>

In [ ]:
class EnergyHistoryHook:
    """Copy graph energy, status, and bounded context after compute."""

    stage = DynamicsStage.AFTER_COMPUTE

    def __init__(self, frequency: int = 1) -> None:
        self.frequency = frequency
        self.rows: list[dict[str, object]] = []
        self.events: list[dict[str, object]] = []
        self.context: dict[str, object] = {}
        self.last_stage: str | None = None

    def on_register(self, host: object) -> None:
        self.events.append({"event": "registered", "host": type(host).__name__})

    def __enter__(self):
        self.events.append({"event": "entered"})
        return self

    def __call__(self, ctx: DynamicsContext, stage: DynamicsStage) -> None:
        batch = ctx.batch
        values = zip(
            batch.system_id.detach().cpu().reshape(-1).tolist(),
            batch.energy.detach().cpu().reshape(-1).tolist(),
            batch.status.detach().cpu().reshape(-1).tolist(),
            strict=True,
        )
        self.rows.extend(
            {
                "step": int(ctx.step_count),
                "system_id": int(system_id),
                "status": int(status),
                "energy (eV)": float(energy),
            }
            for system_id, energy, status in values
        )
        self.context = {
            "workflow": type(ctx.workflow).__name__,
            "batch": type(ctx.batch).__name__,
            "model": type(ctx.model).__name__,
            "step_count": int(ctx.step_count),
            "global_rank": int(ctx.global_rank),
            "converged_mask": None if ctx.converged_mask is None else ctx.converged_mask.cpu().tolist(),
        }
        self.last_stage = stage.name
        self.events.append({"event": "called", "step": int(ctx.step_count), "stage": stage.name})

    def __exit__(self, *_exc_info: object) -> bool:
        self.events.append({"event": "exited"})
        return False

In [ ]:
history_hook = EnergyHistoryHook(frequency=2)
isinstance(history_hook, Hook)

In [ ]:
molecular_nan_guard = dynamics_hooks.NaNDetectorHook(frequency=1)
molecular_hooks = [*molecular_neighbor_hooks, molecular_nan_guard, history_hook]
molecular_host = FIRE2(
    model=model,
    dt=0.01,
    n_steps=4,
    convergence_hook=None,
    hooks=molecular_hooks,
)

In [ ]:
pd.DataFrame(
    {
        "hook": [type(hook).__name__ for hook in molecular_host.hooks],
        "stage": [hook.stage.name for hook in molecular_host.hooks],
        "frequency": [hook.frequency for hook in molecular_host.hooks],
    }
)

## Register the molecular host

Construction has already called `history_hook.on_register(molecular_host)`. During `run()`, the host enters the observer once, dispatches it at steps 0 and 2, and exits it once. The NaN guard appears first at `AFTER_COMPUTE`, so invalid model output stops the run before the observer copies it.

The same three molecules stay in one active batch for four steps. This short run exposes hook behavior only.

In [ ]:
molecular_batch = molecular_host.run(molecular_batch)

In [ ]:
{
    "events": history_hook.events,
    "runtime context": history_hook.context,
    "triggering stage": history_hook.last_stage,
    "batch status": molecular_batch.status.detach().cpu().reshape(-1).tolist(),
    "observer rows": len(history_hook.rows),
}

The host builds this context from the active workflow, batch, model, zero-based step count, rank, and current convergence state. The triggering stage arrives as a separate callback argument. `converged_mask` is `None` because the molecular host was constructed with `convergence_hook=None`.

The event order is `registered`, `entered`, two scheduled calls, then `exited`. Exit confirms resource cleanup. It does not report workflow success.

## Inspect observer-owned history

The observer copied one row per graph at steps 0 and 2. Stable `system_id` values preserve graph ownership after the three variable-size graphs were packed into one `Batch`.

In [ ]:
history = pd.DataFrame(history_hook.rows).merge(
    molecule_table[["system_id", "label"]].rename(columns={"label": "molecule"}),
    on="system_id",
)
history["energy change (meV)"] = 1_000 * (
    history["energy (eV)"]
    - history.groupby("system_id")["energy (eV)"].transform("first")
)
history_summary = history.groupby(["system_id", "molecule"], as_index=False).agg(
    first_energy_eV=("energy (eV)", "first"),
    last_energy_eV=("energy (eV)", "last"),
    records=("step", "size"),
)

In [ ]:
history_summary

## Plot the copied history

What should the relative-energy traces look like if frequency 2 dispatched exactly at steps 0 and 2? The observer records are already CPU-native, so the plotting helper does not retain device tensors.

In [ ]:
history_figure = helpers.plot_energy_history(history)

The scheduled records appear at steps 0 and 2 for every molecule, confirming both frequency gating and graph identity. This short observer demonstration does not establish geometry convergence, relaxation quality, model accuracy, or performance.

## Try it: change the observation schedule

Reuse the custom observer on the quick argon host. Before running the cells, predict its callbacks for `try_frequency = 3` and `n_steps = 5`.

The check uses another ordinary `run()` call. There is no direct callback or fabricated context. Change only `try_frequency`, predict again, and rerun both cells.

In [ ]:
try_frequency = 3
try_hook = EnergyHistoryHook(frequency=try_frequency)
try_hooks = [
    *lj_model.make_neighbor_hooks(),
    dynamics_hooks.MaxForceClampHook(max_force=5.0, frequency=1),
    dynamics_hooks.NaNDetectorHook(frequency=1),
    try_hook,
]
try_host = FIRE2(
    model=lj_model,
    dt=0.002,
    n_steps=5,
    convergence_hook=None,
    hooks=try_hooks,
)

In [ ]:
try_batch = argon_initial.clone()
try_batch.add_key("system_id", [torch.tensor(0)], level="system")
try_batch = try_host.run(try_batch)

expected_try_steps = [0, 3]
observed_try_steps = sorted({row["step"] for row in try_hook.rows})
assert observed_try_steps == expected_try_steps
print("Try it passed:", observed_try_steps)

Frequency 3 selects steps 0 and 3 from the zero-based range 0 through 4. If you change the frequency, update `expected_try_steps` before rerunning so the assertion tests your prediction rather than replacing it.

## Recap

### What you learned

- `Hook` is a structural protocol: provide `stage`, `frequency`, and `__call__`; registration and lifecycle methods are optional.
- The host registers hooks, enters and exits their resources once per run, constructs each `DynamicsContext`, and dispatches matching stages in registration order.
- Frequency gating is zero-based: a callback runs when `step_count % frequency == 0`.
- The active `Batch` owns simulation fields such as `status`. An observer should own copied records when it only needs history.
- The object passed through `convergence_hook=` is a convergence evaluator. The host calls `evaluate(batch)` after `AFTER_STEP` and can use its result for early exit.
- A registered `ConvergenceHook` in `hooks=[...]` can migrate `source_status`/`target_status`, but it does not drive the host's early exit.
- `MaxForceClampHook`, `NaNDetectorHook`, `LoggingHook`, and `SnapshotHook` cover common safety and observation needs before custom code is necessary.
- `__exit__` confirms lifecycle cleanup, not workflow success.

### How we will use this

[Part 05: Base dynamics](../05-base-dynamics/base-dynamics.ipynb) opens the host side of the contract: the step loop, updates, and convergence boundaries you used here. Part 06 builds richer execution and profiling workflows. Its reporting path combines hooks such as `LoggingHook` and `StageTimingHook` with `ReportingOrchestrator` and `RichReporter` rather than turning this lesson into a reporting catalogue.

## Continue with focused examples

Build a domain-specific observer with the official [radial-distribution-function custom hook](https://nvidia.github.io/nvalchemi-toolkit/examples/advanced/02_custom_hook.html). For production guardrails, continue with [safety and monitoring](https://nvidia.github.io/nvalchemi-toolkit/examples/intermediate/05_safety_and_monitoring.html). When terminal reporting or multi-process aggregation becomes the real problem, use [Rich training reporting](https://nvidia.github.io/nvalchemi-toolkit/examples/intermediate/07_rich_training_reporting.html) and [distributed monitoring](https://nvidia.github.io/nvalchemi-toolkit/examples/distributed/02_distributed_monitoring.html).